The below md cell is thought and designed before performing this experiment, assisted by ChatGPT.

# 04 — BTL, Full CDM and Low-Rank CDM on Common Synthetic Data

## Main question

Jab choices BTL, full CDM ya low-rank CDM se generate hoti hain,
toh same training data par fitted BTL, full CDM aur low-rank CDM
unseen choices ko kitna achha predict karte hain?

## Models to fit on every dataset

- BTL/Luce
- Full CDM
- Low-rank CDM: ranks 1, 2 and 3

Every model gets exactly the same train, validation and test observations.

## Three data-generating settings

1. BTL truth:
   Generate choices from a known BTL/Luce parameter vector.
   Include both pairs and triples; the existing pairwise generator alone
   is insufficient for this notebook.

2. Full-CDM truth:
   Start with a nonuniform BTL-like preference pattern and add genuine
   context interactions. At zero context strength, the generator should
   reproduce BTL probabilities on both pairs and triples.

3. Rank-2 CDM truth:
   Generate interactions using known target and context vectors.
   Check whether fitted rank 2 does well, and what happens when we fit
   rank 1, rank 3 and full CDM to the same data.

## Rough settings

- Pilot: n=10, m=5_000, one seed; inspect the complete pipeline.
- Main comparison: n=[10, 20], m=50_000, around 5 independent runs.
- n=40: add only after checking main-run results and runtime.
- Choice-set sizes: (2, 3), with a fixed and reported proportion.
- Base seed: 20260831; log generation, split and optimizer seeds separately.
- Use CPU and torch.float64 initially, consistent with earlier CDM notebooks.
- Split data into train/validation/test; choose rank and optimizer settings
  using validation data, then report test results once.

## Measurements

- Average train, validation and test NLL for every fitted model.
- True generating model's test NLL as an oracle reference.
- Test NLL separately for pair choices and triple choices.
- Parameter RMSE only when comparing compatible, identifiable parameters:
  BTL parameters under BTL truth; off-diagonal CDM interactions under
  CDM truth. Do not compare BTL parameters directly with a CDM matrix.
- Runtime, epochs/updates, final training NLL and stopping reason.
- For low-rank CDM, compare its interaction matrix, not raw target/context
  vectors: different vector representations can produce the same interactions.

## Figures

1. Test NLL of fitted BTL, full CDM and ranks 1/2/3 under each truth.
2. Train versus test NLL: where does extra flexibility help or overfit?
3. Test NLL versus n for the selected generating settings.
4. One direct context example: compare odds of choosing i over j in
   {i, j} versus {i, j, k}. BTL keeps the odds unchanged; a CDM can change them.
5. Rank versus validation/test NLL under rank-2 truth.

## Interpretation questions

- Under BTL truth, does extra CDM flexibility improve prediction,
  make no difference, or overfit?
- Under full-CDM truth, how much does BTL lose on unseen choices?
- Under rank-2 truth, can a compact model match or beat the full CDM?
- Does the best rank change when n changes?
- Are any apparent differences consistent across independent runs?

## Boundary of this notebook

This notebook establishes the shared comparison pipeline and introduces
low-rank CDM. Detailed sweeps over m, sampling schemes and optimizers
belong in Notebook 05. Pairwise-only CDM fitting stays a separate
non-identifiability diagnostic, not the main comparison here.

Reusable generation, fitting and evaluation functions go in src/.
The notebook contains settings, experiment calls, tables, plots and
short conclusions in my own words.

## <span style="color:#1F4E79;">**Unified Synthetic Choice-Model Experiment**</span>
- Use one common setting for BTL-generated and CDM-generated data:
  same number of items, observations, sampled choice sets, split and seed plan.
- Generate both datasets together, then fit each dataset with:
  Choix BTL/Luce and PyTorch full CDM.
- This gives four comparisons in one experiment:
  BTL→BTL, BTL→CDM, CDM→BTL and CDM→CDM.
- Add PyTorch low-rank CDM fits to the same datasets and splits.
- Show the models' results together in one comparison table and
  side-by-side plots. Compare test NLL for all fits, and parameter
  recovery only where the parameters are comparable.

In [2]:
import numpy as np
import pandas as pd
import torch

SEED = 20260831
N_ITEMS = 10
N_OBSERVATIONS = 10_000
SET_SIZE_PROBABILITIES = {2: 0.5, 3: 0.5}
SPLIT_FRACTIONS = {"train": 0.70, "validation": 0.15, "test": 0.15}

DEVICE = torch.device("cpu")
DTYPE = torch.float64

pd.Series({
    "Items": N_ITEMS,
    "Observations per generated dataset": N_OBSERVATIONS,
    "Choice-set sizes": SET_SIZE_PROBABILITIES,
    "Split": SPLIT_FRACTIONS,
    "Seed": SEED,
    "Device": str(DEVICE),
    "PyTorch dtype": str(DTYPE),
}, name="Setting")
from bradley_terry import *
from context_dependent_model import *

| Package | Name | Purpose  |
|---|---|---|
| BTL | `center_parameters` | Centers BTL item parameters so their mean is zero. |
| BTL | `estimate_top1_parameters_with_choix` | Fits BTL with Choix using the chosen item from each choice set, including pairs and triples. |
| BTL | `average_top1_negative_log_likelihood` | Computes average BTL negative log-likelihood, especially on test data. |
| BTL | `parameter_recovery_error` | Computes RMSE between true and fitted BTL parameters. |
| CDM | `generate_cdm_parameters` | Creates the true interaction matrix for synthetic CDM data. |
| CDM | `generate_uniform_choice_sets` | Samples choice sets that we can share between both data generators. |
| CDM | `cdm_choice_probabilities` | Computes CDM choice probabilities for those sets. |
| CDM | `CDMChoiceData` | Holds choice sets, masks, and observed choices. This is a class, not a function. |
| CDM | `optimize_full_cdm_with_adam` | Fits the full CDM using PyTorch and Adam. |
| CDM | `average_cdm_negative_log_likelihood` | Computes average CDM negative log-likelihood, especially on test data. |
| CDM | `center_cdm_parameters` | Sets the interaction matrix diagonal to zero and centers its other entries. |
| CDM | `cdm_parameter_recovery_error` | Computes RMSE between true and fitted CDM interactions. |
| CDM | `generate_cdm_choice_data` | Generates CDM data, but samples its own choice sets; it cannot directly reuse our shared sets. |
| CDM | `split_cdm_choice_data` | Makes a train/test split; it does not currently provide a validation split. |

In [5]:
# Save this run's settings, true parameters, and generated choices.
from pathlib import Path
import json

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "docs" / "thesis" / "images").is_dir()
)
RUN_ID = f"n{N_ITEMS}_m{N_OBSERVATIONS}_seed{SEED}_scale{PARAMETER_SCALE:g}"

FIGURE_DIR = PROJECT_ROOT / "docs/thesis/images/experiments/context_dependent_model/unified" / RUN_ID
TABLE_DIR = PROJECT_ROOT / "docs/thesis/tables/experiments/context_dependent_model/unified" / RUN_ID
LOG_DIR = PROJECT_ROOT / "results/experiments/04_unified_synthetic" / RUN_ID

for folder in (FIGURE_DIR, TABLE_DIR, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

settings = {
    "n_items": N_ITEMS,
    "observations_per_dataset": N_OBSERVATIONS,
    "parameter_scale": PARAMETER_SCALE,
    "set_size_probabilities": SET_SIZE_PROBABILITIES,
    "split_fractions": SPLIT_FRACTIONS,
    "seed": SEED,
    "device": str(DEVICE),
    "dtype": str(DTYPE),
}
(LOG_DIR / "settings.json").write_text(
    json.dumps(settings, indent=2), encoding="utf-8"
)

pd.DataFrame({
    "choice_set": [json.dumps(items) for items in shared_choice_sets],
    "set_size": [len(items) for items in shared_choice_sets],
    "btl_choice": [winner for winner, _ in btl_observations],
    "cdm_choice": cdm_choice_data.chosen_items.cpu().tolist(),
}).to_csv(LOG_DIR / "generated_choices.csv", index=False)

pd.Series(true_btl_parameters, name="parameter").rename_axis("item").to_csv(
    LOG_DIR / "true_btl_parameters.csv"
)
pd.DataFrame(true_cdm_parameters.cpu().numpy()).rename_axis("target_item").to_csv(
    LOG_DIR / "true_cdm_parameters.csv"
)

pd.Series({
    "Figures": str(FIGURE_DIR),
    "Thesis tables": str(TABLE_DIR),
    "Data and settings": str(LOG_DIR),
})

Figures              /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...
Thesis tables        /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...
Data and settings    /Users/zsres/Documents/M.Tech/Nitesh-Kumar-Sha...
dtype: object

In [4]:
# Generate BTL and CDM choices on the same sampled choice sets.
PARAMETER_SCALE = 0.4

true_btl_parameters = center_parameters(
    np.random.default_rng(SEED).normal(0, PARAMETER_SCALE, N_ITEMS)
)
true_cdm_parameters = generate_cdm_parameters(
    N_ITEMS, parameter_scale=PARAMETER_SCALE,
    random_seed=SEED + 1, device=DEVICE, dtype=DTYPE,
)

choice_sets, choice_mask = generate_uniform_choice_sets(
    N_ITEMS, N_OBSERVATIONS, tuple(SET_SIZE_PROBABILITIES),
    random_seed=SEED + 2, device=DEVICE,
)
shared_choice_sets = [
    tuple(row[mask].tolist())
    for row, mask in zip(choice_sets, choice_mask)
]

btl_observations = generate_top1_choice_data_on_sets(
    true_btl_parameters, shared_choice_sets,
    np.random.default_rng(SEED + 3),
)
btl_choice_data = CDMChoiceData(
    choice_sets=choice_sets,
    choice_mask=choice_mask,
    chosen_positions=torch.tensor(
        [items.index(winner) for winner, items in btl_observations],
        dtype=torch.long, device=DEVICE,
    ),
)

cdm_choice_data = generate_cdm_choice_data_on_sets(
    true_cdm_parameters, choice_sets, choice_mask,
    random_seed=SEED + 4,
)
cdm_observations = list(zip(
    cdm_choice_data.chosen_items.cpu().tolist(),
    shared_choice_sets,
))

pd.DataFrame({
    "Choice set": shared_choice_sets[:5],
    "BTL choice": [winner for winner, _ in btl_observations[:5]],
    "CDM choice": cdm_choice_data.chosen_items[:5].cpu().tolist(),
})

,Choice set,BTL choice,CDM choice
0,"(7, 9)",7,9
1,"(1, 3)",1,1
2,"(2, 9, 1)",2,1
3,"(4, 3)",4,3
4,"(7, 3, 5)",7,3
